# 602 SPP: source-input-fair compact LSTM

Training and inference use the same chronological `DEMAND(addr)` and `CACHE_FILL(evicted_addr)` encoder. Source SPP actions are labels/comparator replay only. The decoder has no probability threshold, degree cap, fixed candidate/page-offset table, same-page rule, or normal-policy state.


In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'
TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy()
env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else:
        subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally:
    if os.path.exists(ASKPASS):
        os.unlink(ASKPASS)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='602_offline_lstm_spp_compact_hurdle_free_running_v1_seed7'
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_602_spp/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'
os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'
uploaded=files.upload()
assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'
pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle:
    handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1)
    item=item.lstrip('*')
    observed=hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()
    assert observed==expected,(item,observed,expected)
print('verified',archive)


In [ ]:
TRACE='602.gcc_s-734B'
POLICY='spp'
ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
    for path in items.values():
        assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':'spp_source_input_compact_hurdle_delta_fill_free_running_v1','event_logger_schema':'602_spp_causal_trigger_fill_v1','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'fill_lead_cutoff_used':False,'inference_policy_hardcodes_used':False,'complete_neural_action_space':True}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}
assert not bad,bad
fields=['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']
assert manifest['training_runtime_fields']==fields==manifest['inference_runtime_fields']
SCRIPT=f'{REPO}/formal_NN_training/experiments/602_offline_lstm_spp/python/train_and_offline_infer.py'
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'


In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT):
    shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[
 {'tag':'independent_delta_spp_lstm_h8','size':8,'pair':'p0','parameters':2865},
 {'tag':'independent_delta_spp_lstm_h16','size':16,'pair':'p1','parameters':6609},
 {'tag':'independent_delta_spp_lstm_h32','size':32,'pair':'p2','parameters':16785},
 {'tag':'independent_delta_spp_lstm_h64','size':64,'pair':'p3','parameters':47889},
 {'tag':'independent_delta_spp_lstm_h128','size':128,'pair':'p4','parameters':153105},
]
SWEEP=[]
for spec in SPECS:
    out=f"{LOCAL_OUTPUT}/{spec['tag']}"
    cmd=[sys.executable,SCRIPT,'--policy',POLICY]
    for role in ROLES:
        cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
    cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family','lstm','--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--epochs','8','--chunk-len','1024','--accumulate-chunks','16']
    print('\nTraining',spec['tag'],' '.join(cmd),flush=True)
    subprocess.run(cmd,check=True)
    meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
    expected={'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm','parameter_count':spec['parameters'],'model_revision':'compact_hurdle_autoregressive_gmm_fill_v1','training_state_mode':'chronological_stateful_tbptt','inference_history_mode':'fresh_state_then_complete_train_guard_eval_chronology','matched_normal_prefetcher':POLICY,'neural_role':'standalone_direct_action_prefetcher','same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'source_decision_effective_external_input':['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'fill_lead_cutoff_used':False,'handcrafted_semantic_features_used':False,'manual_loss_weights_used':False,'training_regularization_used':False,'inference_policy_hardcodes_used':False,'learned_request_count':True,'experiment_revision':'spp_source_input_compact_hurdle_delta_fill_free_running_v1','decoder_free_running_self_test':'PASS'}
    bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}
    assert not bad,bad
    fields=['callback_kind','invoke_prefetcher.addr','cache_fill.evicted_addr']
    assert meta['training_runtime_fields']==fields==meta['inference_runtime_fields']
    hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
    assert len(hashes)==1 and isinstance(next(iter(hashes)),str) and len(next(iter(hashes)))==64,hashes
    SWEEP.append({k:meta[k] for k in ('model_tag','model_size','architecture_pair_id','parameter_count','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT):
    shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'revision':'spp_source_input_compact_hurdle_delta_fill_free_running_v1','points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))


In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
    for item in pathlib.Path(OUTPUT_ROOT).iterdir():
        archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')


Download the output archive and copy it back to the same Sacramento run. Replay compares offline SPP and every LSTM point through the same fill-preserving transport.
